In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.geometry import box
from shapely.ops import unary_union
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.features import geometry_mask
from post_processing_functions import load_traffic_centers  # uses EPSG:28992 buffering

# -------------------
# Configuration
# -------------------
traffic_centers_file = r"P:\bovenregionale-stresstest-hwn\Data\Traffic_centrals\Traffic_centers.xlsx"
areas_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
hazard_base = Path(r"P:\bovenregionale-stresstest-hwn\Analysis")
output_directory = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\ByBuffer")
output_directory.mkdir(parents=True, exist_ok=True)

# Buffer sizes (meters)
buffer_distances = [5, 10, 50, 100, 200]

# -------------------
# Helpers
# -------------------
def pick_region_name_column(gdf: gpd.GeoDataFrame) -> str:
    candidates = ["name"]
    for c in candidates:
        if c in gdf.columns:
            return c
    raise KeyError(f"Could not find a region-name column in Areas.gpkg. Columns: {list(gdf.columns)}")

def list_region_rasters(region: str) -> list[Path]:
    hz_dir = hazard_base / region / "Inputs" / "static" / "hazard"
    if not hz_dir.exists():
        return []
    patt = "*max_wd_merge_clipNL*"
    files = list(hz_dir.glob(patt + ".tif")) + list(hz_dir.glob(patt + ".tiff")) + list(hz_dir.glob(patt))
    print(f"Found {len(files)} rasters for region '{region}' in {hz_dir}")
    # Deduplicate and keep only files
    files = [f for f in dict.fromkeys(files) if f.is_file()]
    return files

def datasets_intersecting_bounds(paths: list[Path], bounds) -> list[rasterio.io.DatasetReader]:
    """
    Open only those datasets whose bounds intersect the given bounds (minx, miny, maxx, maxy).
    """
    minx, miny, maxx, maxy = bounds
    bbox_poly = box(minx, miny, maxx, maxy)
    ds_list = []
    for p in paths:
        try:
            ds = rasterio.open(p)
            ds_bbox_poly = box(*ds.bounds)
            if ds_bbox_poly.intersects(bbox_poly):
                ds_list.append(ds)
            else:
                ds.close()
        except Exception:
            # Skip unreadable rasters
            pass
    return ds_list

def stats_from_rasters_for_geom(geom, rasters: list[Path], nodata_fallback= -9999.0) -> dict:
    """
    Mosaic rasters within geom bounds (per-pixel max), then compute mean and max within the polygon.
    Returns dict with mean/max, pixel counts and which rasters were used. If no overlap: sentinel -9999.
    """
    if not rasters:
        return {
            "max_flood_depth": -9999.0,
            "mean_flood_depth": -9999.0,
            "pixel_count": 0,
            "flooded_pixels": 0,
            "rasters_used": 0
        }

    bounds = geom.bounds
    # Filter to rasters intersecting the buffer BBOX to avoid heavy reads
    ds_list = datasets_intersecting_bounds(rasters, bounds)
    if not ds_list:
        return {
            "max_flood_depth": -9999.0,
            "mean_flood_depth": -9999.0,
            "pixel_count": 0,
            "flooded_pixels": 0,
            "rasters_used": 0
        }

    try:
        # Use smallest pixel size among sources
        pix_sizes = [min(abs(ds.transform.a), abs(ds.transform.e)) for ds in ds_list]
        res = float(min(pix_sizes))
        # Unify nodata across datasets (fallback if none set)
        nd = next((ds.nodata for ds in ds_list if ds.nodata is not None), nodata_fallback)

        # Mosaic with per-pixel max limited to buffer bounds for speed
        # Mosaic with per-pixel max limited to buffer bounds for speed
        mosaic_arr, mosaic_transform = rio_merge(
            ds_list,
            bounds=bounds,
            nodata=nd,
            res=res,
            method="max"
        )
        # Check for empty mosaic
        if mosaic_arr.shape[1] == 0 or mosaic_arr.shape[2] == 0:
            print(f"Warning: Mosaic for buffer at {bounds} is empty (no raster overlap).")
            return {
                "max_flood_depth": -9999.0,
                "mean_flood_depth": -9999.0,
                "pixel_count": 0,
                "flooded_pixels": 0,
                "rasters_used": len(ds_list)
            }
        band = mosaic_arr[0].astype("float32", copy=False)

        # Mask to the exact polygon
        mask = geometry_mask(
            [geom.__geo_interface__],
            transform=mosaic_transform,
            invert=True,
            out_shape=band.shape
        )

        valid = mask & np.isfinite(band) & (band != nd)
        if not np.any(valid):
            return {
                "max_flood_depth": -9999.0,
                "mean_flood_depth": -9999.0,
                "pixel_count": 0,
                "flooded_pixels": 0,
                "rasters_used": len(ds_list)
            }

        values = band[valid]
        max_val = float(values.max()) if values.size else -9999.0
        mean_val = float(values.mean()) if values.size else -9999.0
        # flooded_pixels: count > 0 depths
        flooded_pixels = int((values > 0).sum())
        return {
            "max_flood_depth": max_val,
            "mean_flood_depth": mean_val,
            "pixel_count": int(values.size),
            "flooded_pixels": flooded_pixels,
            "rasters_used": len(ds_list)
        }
    finally:
        for ds in ds_list:
            try:
                ds.close()
            except Exception:
                pass

def analyze_for_buffers(gdf_buffers: gpd.GeoDataFrame, regions_gdf: gpd.GeoDataFrame, region_col: str) -> gpd.GeoDataFrame:
    """
    For each buffer geometry, find overlapping regions, gather their rasters,
    mosaic if >1 region, then compute stats (mean, max) within the buffer.
    """
    # Ensure same CRS
    if regions_gdf.crs != gdf_buffers.crs:
        regions_gdf = regions_gdf.to_crs(gdf_buffers.crs)

    # Spatial join to attach possible multiple regions; aggregate to list per buffer
    join = gpd.sjoin(
        gdf_buffers[['geometry']].copy(),
        regions_gdf[[region_col, 'geometry']].copy(),
        how='left',
        predicate='intersects'
    )
    # Build mapping: buffer index -> list of unique regions
    region_lists = (join
                .groupby(join.index)[region_col]
                .apply(lambda s: sorted({x for x in s.dropna().tolist()}))
                .reindex(gdf_buffers.index)
                .apply(lambda x: x if isinstance(x, list) else [])
               )
    # Compute stats per buffer
    results = []
    for idx, geom in gdf_buffers.geometry.items():
        regions = region_lists.loc[idx]
        # Collect rasters across all overlapping regions
        raster_paths = []
        for r in regions:
            raster_paths.extend(list_region_rasters(r))
        # Run stats
        stats = stats_from_rasters_for_geom(geom, raster_paths)
        results.append({
            "index": idx,
            "regions_overlap": "; ".join(regions) if regions else "",
            **stats
        })

    stats_df = pd.DataFrame(results).set_index("index")
    out_gdf = gdf_buffers.join(stats_df, how="left")
    return out_gdf

# -------------------
# Run for each buffer size
# -------------------
summaries = {}
# Load regions polygons (first layer)
regions_gdf = gpd.read_file(areas_gpkg)
region_col = pick_region_name_column(regions_gdf)
print(f"Using region column: {region_col}")

for buffer_distance in buffer_distances:
    print(f"\n=== Running point-by-point analysis for buffer: {buffer_distance} m ===")
    gdf_buffers = load_traffic_centers(traffic_centers_file, buffer_distance)  # EPSG:28992
    out_gdf = analyze_for_buffers(gdf_buffers, regions_gdf, region_col)

    # Summary
    valid_mask = out_gdf['max_flood_depth'] != -9999
    summary = {
        "total_centers": int(len(out_gdf)),
        "with_region_overlap": int((out_gdf['regions_overlap'] != "").sum()),
        "with_data": int(valid_mask.sum()),
        "with_flooding": int((out_gdf['max_flood_depth'] > 0).sum()),
        "max_found": float(out_gdf['max_flood_depth'].max()) if len(out_gdf) else -9999.0,
        "mean_of_max_over_centers": float(out_gdf.loc[valid_mask, 'max_flood_depth'].mean()) if valid_mask.any() else -9999.0
    }
    summaries[buffer_distance] = summary
    print(summary)

    # Save per buffer size
    gpkg_path = output_directory / f"traffic_centers_buffers_{buffer_distance}m_bybuffer.gpkg"
    xlsx_path = output_directory / f"traffic_centers_buffers_{buffer_distance}m_bybuffer.xlsx"
    if gpkg_path.exists():
        gpkg_path.unlink()
    out_gdf.to_file(gpkg_path, driver="GPKG")
    out_gdf.drop(columns="geometry").to_excel(xlsx_path, index=False)
    print(f"Saved: {gpkg_path}\nSaved: {xlsx_path}")

summaries

Using region column: name

=== Running point-by-point analysis for buffer: 5 m ===
Loading traffic centers...
Loaded 6 traffic centers with 5m buffers
Found 2 rasters for region 'ARK-NZK' in P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Inputs\static\hazard
Found 2 rasters for region 'Noord-Brabant Oost' in P:\bovenregionale-stresstest-hwn\Analysis\Noord-Brabant Oost\Inputs\static\hazard
Found 2 rasters for region 'ARK-NZK' in P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Inputs\static\hazard
Found 2 rasters for region 'Vallei en Veluwe' in P:\bovenregionale-stresstest-hwn\Analysis\Vallei en Veluwe\Inputs\static\hazard
Found 2 rasters for region 'ARK-NZK' in P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Inputs\static\hazard
Found 2 rasters for region 'Noord-Westelijke Delta' in P:\bovenregionale-stresstest-hwn\Analysis\Noord-Westelijke Delta\Inputs\static\hazard
{'total_centers': 6, 'with_region_overlap': 6, 'with_data': 1, 'with_flooding': 1, 'max_found': 2.352730274200439

{5: {'total_centers': 6,
  'with_region_overlap': 6,
  'with_data': 1,
  'with_flooding': 1,
  'max_found': 2.3527302742004395,
  'mean_of_max_over_centers': 2.3527302742004395},
 10: {'total_centers': 6,
  'with_region_overlap': 6,
  'with_data': 2,
  'with_flooding': 2,
  'max_found': 2.4391074180603027,
  'mean_of_max_over_centers': 1.24047826603055},
 50: {'total_centers': 6,
  'with_region_overlap': 6,
  'with_data': 2,
  'with_flooding': 2,
  'max_found': 5.036980152130127,
  'mean_of_max_over_centers': 2.8112900853157043},
 100: {'total_centers': 6,
  'with_region_overlap': 6,
  'with_data': 2,
  'with_flooding': 2,
  'max_found': 6.276547908782959,
  'mean_of_max_over_centers': 3.4597739577293396},
 200: {'total_centers': 6,
  'with_region_overlap': 6,
  'with_data': 5,
  'with_flooding': 5,
  'max_found': 6.276547908782959,
  'mean_of_max_over_centers': 1.6557932853698731}}

In [ ]:
from post_processing_functions import load_traffic_centers, process_all_regions, save_results

def run_flood_analysis_multi_buffer():
    # Configuration
    traffic_centers_file = r"P:\bovenregionale-stresstest-hwn\Data\Traffic_centrals\Traffic_centers.xlsx"
    #hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Data\Hazard_maps\Hazard_maps-in_use" #change this to add regions from analysis
    hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Analysis"
    output_directory = r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis"
    region_list = ["ARK-NZK", "Vallei en Veluwe", "Noord-Westelijke Delta", "Noord-Brabant Oost","Achterhoek"]
    buffer_distances = [5, 10, 50, 100, 200]  # meters

    all_summaries = {}

    for buffer_distance in buffer_distances:
        print(f"\n=== Running analysis for buffer distance: {buffer_distance}m ===")
        # Step 1: Load and buffer traffic centers
        gdf_buffered = load_traffic_centers(traffic_centers_file, buffer_distance)

        # Step 2: Process all regions and flood maps
        gdf_with_floods, all_results = process_all_regions(gdf_buffered, region_list, hazard_maps_base_path)

        # Step 3: Save results in a subfolder for each buffer distance
        buffer_output_dir = f"{output_directory}\\buffer_{buffer_distance}m"
        gpkg_path, excel_path = save_results(gdf_with_floods, buffer_output_dir)

        # Generate summary statistics
        summary_stats = {
            'total_traffic_centers': len(gdf_with_floods),
            'centers_with_flood_data': (gdf_with_floods['max_flood_depth'] != -9999).sum(),
            'centers_with_flooding': (gdf_with_floods['max_flood_depth'] > 0).sum(),
            'max_flood_depth_found': gdf_with_floods['max_flood_depth'].max(),
            'mean_flood_depth': gdf_with_floods[gdf_with_floods['max_flood_depth'] != -9999]['max_flood_depth'].mean(),
            'gpkg_path': gpkg_path,
            'excel_path': excel_path
        }
        all_summaries[buffer_distance] = summary_stats

        #print(f"\nAnalysis complete for buffer {buffer_distance}m!")
        #print(f"Processed {len(all_results)} flood maps across {len(region_list)} regions.")

    return all_summaries

summaries = run_flood_analysis_multi_buffer()